# Analyze the distribution of the SCOTUS expert annotated data

I cleaned the SCOTUS expert annotated data (round 1) in a Google Sheet and normalized some of the expert comments and clean-up some of the labels (there were a few labels missing which I have now added). This is to take a look at the distribution of the annotated data

# Import Libraries

In [1]:
import numpy as np
import pandas as pd

# Load data

In [2]:
df = pd.read_csv("0.annotated_round1.csv")
df.head()

,citing_cluster_id,citing_url,cited_cluster_id,cited_url,cited_case_name_short,cited_case_name,cited_citations,expert,expert_label,expert_notes
0,105846,https://www.courtlistener.com/opinion/105846/m...,105837.0,https://www.courtlistener.com/opinion/105837/k...,Kermarec,Kermarec v. Compagnie Generale Transatlantique,"['1959 U.S. LEXIS 1769', '358 U.S. 625', '79 S...",Catherine McCarthy,Cited by,NaN
1,105846,https://www.courtlistener.com/opinion/105846/m...,245707.0,https://www.courtlistener.com/opinion/245707/j...,NaN,James E. McDaniel Libelant-Appellant v. The M/...,"['257 F.2d 538', '1958 U.S. App. LEXIS 5333']",Catherine McCarthy,Vacated and remanded by,case on appeal not directly cited
2,100915,https://www.courtlistener.com/opinion/100915/m...,96357.0,https://www.courtlistener.com/opinion/96357/so...,South Carolina,South Carolina v. United States,"['3 A.F.T.R. (P-H) 2775', '1905 U.S. LEXIS 991...",Catherine McCarthy,Cited by,NaN
3,100915,https://www.courtlistener.com/opinion/100915/m...,8608009.0,https://www.courtlistener.com/opinion/8608009/...,NaN,Missouri Pacific Railroad v. United States,"['1925 WL 2700', '1925 U.S. Ct. Cl. LEXIS 568'...",Catherine McCarthy,Affirmed by,NaN
4,105845,https://www.courtlistener.com/opinion/105845/a...,105825.0,https://www.courtlistener.com/opinion/105825/r...,Romero,Romero v. International Terminal Operating Co.,"['1959 U.S. LEXIS 1747', '358 U.S. 354', '79 S...",Catherine McCarthy,Cited by,NaN


# EDA

In [3]:
len(df)

7619

In [4]:
assert df["citing_cluster_id"].isnull().sum() == 0

In [5]:
assert df["expert_label"].isnull().sum() == 0

In [6]:
df["cited_cluster_id"].isnull().sum()

np.int64(12)

In [7]:
df[df["cited_cluster_id"].isnull()][["citing_cluster_id", "expert_label", "expert_notes"]]

,citing_cluster_id,expert_label,expert_notes
19,99467,Affirmed by,case on appeal not directly cited
52,99731,Reversed by,case on appeal not directly cited
117,97004,Reversed by,case on appeal not directly cited
152,98518,Reversed by,case on appeal not directly cited
272,102174,Affirmed by,case on appeal not directly cited
571,97396,Reversed by,case on appeal not directly cited
1120,101642,Affirmed by,case on appeal not directly cited
1455,94610,Reversed by,case on appeal not directly cited
2268,99541,Affirmed by,case on appeal not directly cited
3574,2518018,Affirmed by,case on appeal not directly cited


In [8]:
df["citing_cluster_id"].nunique()

182

In [9]:
df["expert_label"].value_counts().sort_index()

expert_label
Abrogated as recognized by                  6
Abrogated by                               37
Affirmed by                                92
Affirmed in part; Reversed in part by       3
Cited by                                 6808
Criticized as recognized by                 1
Criticized by                              55
Declined to follow by                      27
Disapproved by                             36
Dismissed by                                2
Distinguished as recognized by              2
Distinguished by                          345
Limited by                                 22
Overruled as recognized by                 14
Overruled by                               31
Questioned by                              29
Remanded as recognized by                   1
Remanded by                                 2
Reversed and remanded by                    6
Reversed as recognized by                   3
Reversed by                                76
Vacated and remanded 

In [10]:
df["expert_label"].value_counts()

expert_label
Cited by                                 6808
Distinguished by                          345
Affirmed by                                92
Reversed by                                76
Criticized by                              55
Abrogated by                               37
Disapproved by                             36
Overruled by                               31
Questioned by                              29
Declined to follow by                      27
Limited by                                 22
Vacated and remanded by                    17
Overruled as recognized by                 14
Abrogated as recognized by                  6
Reversed and remanded by                    6
Vacated by                                  3
Affirmed in part; Reversed in part by       3
Reversed as recognized by                   3
Dismissed by                                2
Distinguished as recognized by              2
Remanded by                                 2
Criticized as recogni

In [11]:
severity = {"Stop": ['Reversed by', 'Reversed and remanded by', 'Vacated and remanded by', 'Vacated by', 'Overruled by', 'Abrogated by', 'Questioned by', 
                    'Reversed as recognized by', 'Reversed and remanded as recognized by', 'Vacated and remanded as recognized by', 'Vacated as recognized by', 'Overruled as recognized by', 'Abrogated as recognized by', 'Questioned as recognized by'],
            "Warning": ['Affirmed in part; Reversed in part by', 'Disapproved by', 'Limited by',
                        'Affirmed in part; Reversed in part as recognized by', 'Disapproved as recognized by', 'Limited as recognized by'],
            "Caution": ['Remanded by', 'Criticized by', 'Distinguished by', 'Declined to follow by',
                        'Remanded as recognized by', 'Criticized as recognized by', 'Distinguished as recognized by', 'Declined to follow as recognized by'],
            "Neutral": ['Dismissed by', 'Affirmed by', 'Cited by',
                        'Dismissed as recognized by', 'Affirmed as recognized by']
            }

label_to_severity = {label: category for category, labels in severity.items() for label in labels}

In [12]:
direction = {"Direct History": ['Reversed by', 'Reversed and remanded by', 'Vacated and remanded by', 'Vacated by',
                                'Affirmed in part; Reversed in part by', 
                                'Remanded by', 
                                'Dismissed by', 'Affirmed by'],
             "Citing Reference": ['Overruled by', 'Abrogated by', 'Questioned by',
                                  'Disapproved by', 'Limited by',
                                  'Criticized by', 'Distinguished by', 'Declined to follow by',
                                  'Cited by'],
             "Related Reference": ['Reversed as recognized by', 'Reversed and remanded as recognized by', 'Vacated and remanded as recognized by', 'Vacated as recognized by', 'Overruled as recognized by', 'Abrogated as recognized by', 'Questioned as recognized by',
                                   'Affirmed in part; Reversed in part as recognized by', 'Disapproved as recognized by', 'Limited as recognized by',
                                   'Remanded as recognized by', 'Criticized as recognized by', 'Distinguished as recognized by', 'Declined to follow as recognized by',
                                   'Dismissed as recognized by', 'Affirmed as recognized by']
            }

label_to_direction = {label: category for category, labels in direction.items() for label in labels}

In [13]:
assert set([each for sublist in severity.values() for each in sublist]) == set([each for sublist in direction.values() for each in sublist])

In [14]:
df["expert_severity"] = df["expert_label"].map(label_to_severity)
df["expert_direction"] = df["expert_label"].map(label_to_direction)

In [15]:
df["expert_severity"].value_counts()

expert_severity
Neutral    6902
Caution     433
Stop        223
Warning      61
Name: count, dtype: int64

In [16]:
df["expert_direction"].value_counts()

expert_direction
Citing Reference     7390
Direct History        201
Related Reference      28
Name: count, dtype: int64

In [17]:
df["expert"].value_counts()

expert
Catherine McCarthy    1205
Miriam Marks          1176
Rebecca Pressman       769
Andrew Nordick         766
Aimee Self Pittman     764
Nor Ortiz              419
Stephanie Grace        357
Shay Elbaum            356
Alicia Diaz Wrest      354
Ursula Gorham          351
Zvi Rosen              347
Paul Menair            345
Grace Lo               176
Andrew Green           119
James Alcorn            60
Ben Ridgway             35
Ellen Kim               12
Huihui Xu                8
Name: count, dtype: int64